# CatBoost v2: подбор rsm

Задача ноутбука — проверить случайный отбор признаков при выборе разбиений CatBoost. Набор `enhanced_v2` из 216 признаков и все ранее выбранные параметры остаются неизменными.

Текущая reference-конфигурация: `depth=6`, `l2_leaf_reg=10`, `random_strength=0.5`, `rsm=1.0`. Её global OOF RMSLE равен `1.731068`. Проверяются `rsm=0.85`, `0.70` и `0.50`.

## 0. Режим запуска

`RUN_SCREENING=True` обучает кандидатов на раннем и позднем временных фолдах. Каждый кандидат сохраняется немедленно в отдельный CSV. `RUN_FULL_VALIDATION=True` проверяет победителя screening на всех четырёх фолдах. После выполнения эксперимента оба флага выключаются.

In [1]:
RUN_SCREENING = False
RUN_FULL_VALIDATION = False

SCREENING_ANCHORS = ('2025-10-22', '2026-01-14')

## 1. Импорты и данные

Используются готовые v2-срезы и те же временные фолды. Признаки заново не строятся.

In [2]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_RSM_DIR,
    CATBOOST_TUNING_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    VALIDATION_ANCHORS,
    ensure_output_dirs,
)
from src.experiments import run_temporal_experiment
from src.features import load_snapshots
from src.models import make_validation_model
from src.validation import feature_columns, make_temporal_folds, rmsle

ensure_output_dirs()
snapshots = load_snapshots(CATBOOST_V2_SNAPSHOT_DIR, kind='train')
historical_anchors = sorted(snapshots)
features = feature_columns(snapshots[historical_anchors[0]])
assert len(features) == 216

all_folds = make_temporal_folds(historical_anchors, VALIDATION_ANCHORS)
screening_folds = [
    fold for fold in all_folds
    if fold.validation_anchor.isoformat() in SCREENING_ANCHORS
]
print(f'Признаков: {len(features)}')
print('Screening holdout:', [fold.validation_anchor for fold in screening_folds])

Признаков: 216
Screening holdout: [datetime.date(2025, 10, 22), datetime.date(2026, 1, 14)]


## 2. Reference и кандидаты

При 216 признаках значения `0.85`, `0.70` и `0.50` дают CatBoost доступ примерно к 184, 151 и 108 признакам при очередном выборе разбиения. Reference `rsm=1.0` использует все признаки и не переобучается.

In [3]:
FIXED_PARAMS = {
    'depth': 6,
    'l2_leaf_reg': 10.0,
    'random_strength': 0.5,
}
RSM_CANDIDATES = {
    'rsm_0_85': 0.85,
    'rsm_0_70': 0.70,
    'rsm_0_50': 0.50,
}

reference_metrics = pd.read_csv(
    CATBOOST_TUNING_DIR / 'depth_winner_multifold_metrics.csv'
)
reference_oof = pd.read_parquet(
    CATBOOST_TUNING_DIR / 'depth_winner_oof.parquet'
)
reference_oof_rmsle = rmsle(
    reference_oof['target'].to_numpy(),
    reference_oof['prediction'].to_numpy(),
)
display(pd.DataFrame([
    {
        'candidate': name,
        'rsm': value,
        'approximately_available_features': round(len(features) * value),
    }
    for name, value in RSM_CANDIDATES.items()
]))
print(f'Reference global OOF RMSLE: {reference_oof_rmsle:.6f}')

,candidate,rsm,approximately_available_features
0,rsm_0_85,0.85,184
1,rsm_0_70,0.70,151
2,rsm_0_50,0.50,108


Reference global OOF RMSLE: 1.731068


## 3. Screening на двух временных фолдах

Кандидаты обучаются последовательно. После каждого кандидата его две строки метрик записываются в отдельный файл. При повторном запуске существующие результаты загружаются и не пересчитываются.

In [4]:
screening_parts = []
for candidate_name, rsm_value in RSM_CANDIDATES.items():
    candidate_path = CATBOOST_RSM_DIR / f'{candidate_name}_screening_metrics.csv'
    if RUN_SCREENING and not candidate_path.exists():
        candidate_metrics, _ = run_temporal_experiment(
            snapshots=snapshots,
            folds=screening_folds,
            features=features,
            model_factory=partial(
                make_validation_model,
                **FIXED_PARAMS,
                rsm=rsm_value,
            ),
            keep_oof=False,
            label=candidate_name,
        )
        candidate_metrics['rsm'] = rsm_value
        candidate_metrics.to_csv(candidate_path, index=False)
        print(f'Сохранено: {candidate_path.name}')
    elif candidate_path.exists():
        candidate_metrics = pd.read_csv(candidate_path)
        print(f'Загружено: {candidate_path.name}')
    else:
        raise FileNotFoundError(
            f'Нет {candidate_path.name}. Установите RUN_SCREENING=True.'
        )
    screening_parts.append(candidate_metrics)

screening_metrics = pd.concat(screening_parts, ignore_index=True)
screening_metrics.to_csv(
    CATBOOST_RSM_DIR / 'screening_metrics.csv', index=False
)

reference_screening = reference_metrics[
    reference_metrics['validation_anchor'].isin(SCREENING_ANCHORS)
].copy()
reference_screening['experiment'] = 'rsm_1_reference'
reference_screening['rsm'] = 1.0
screening_comparison = pd.concat(
    [screening_metrics, reference_screening[screening_metrics.columns]],
    ignore_index=True,
)
display(screening_comparison.sort_values(['validation_anchor', 'catboost_rmsle']))

[rsm_0_85] holdout 2025-10-22


0:	learn: 2.2813886	test: 2.2959746	best: 2.2959746 (0)	total: 445ms	remaining: 13m 19s


200:	learn: 1.6937947	test: 1.7172773	best: 1.7172773 (200)	total: 1m 6s	remaining: 8m 47s


400:	learn: 1.6893189	test: 1.7155803	best: 1.7155803 (400)	total: 2m 2s	remaining: 7m 5s


600:	learn: 1.6859523	test: 1.7151202	best: 1.7151192 (599)	total: 2m 53s	remaining: 5m 46s


...

1400:	learn: 1.6973054	test: 1.7051241	best: 1.7051135 (1348)	total: 12m 44s	remaining: 3m 37s


1600:	learn: 1.6960018	test: 1.7050349	best: 1.7050127 (1592)	total: 14m 6s	remaining: 1m 45s


1799:	learn: 1.6947108	test: 1.7051159	best: 1.7050012 (1636)	total: 15m 27s	remaining: 0us

bestTest = 1.705001196
bestIteration = 1636

Shrink model to first 1637 iterations.


  RMSLE=1.704992; baseline=2.195065
Сохранено: rsm_0_50_screening_metrics.csv


,experiment,validation_anchor,n_features,n_train_rows,catboost_rmsle,baseline_rmsle,improvement,best_iteration,rsm
4,rsm_0_50,2025-10-22,216,750000,1.714427,2.158052,0.443625,1123,0.50
0,rsm_0_85,2025-10-22,216,750000,1.714715,2.158052,0.443337,1307,0.85
2,rsm_0_70,2025-10-22,216,750000,1.714723,2.158052,0.443329,978,0.70
6,rsm_1_reference,2025-10-22,216,750000,1.714746,2.158052,0.443306,1302,1.00
7,rsm_1_reference,2026-01-14,216,1500000,1.703598,2.195065,0.491467,722,1.00
5,rsm_0_50,2026-01-14,216,1500000,1.704992,2.195065,0.490072,1636,0.50
3,rsm_0_70,2026-01-14,216,1500000,1.705229,2.195065,0.489836,953,0.70
1,rsm_0_85,2026-01-14,216,1500000,1.705607,2.195065,0.489458,1084,0.85


## 4. Выбор победителя screening

Основной критерий — средний RMSLE раннего и позднего фолдов. Дополнительно контролируются худший RMSLE и направление изменения на каждой дате.

In [5]:
screening_summary = (
    screening_comparison.groupby(['experiment', 'rsm'], as_index=False)
    .agg(
        mean_rmsle=('catboost_rmsle', 'mean'),
        worst_rmsle=('catboost_rmsle', 'max'),
        mean_best_iteration=('best_iteration', 'mean'),
    )
    .sort_values(['mean_rmsle', 'worst_rmsle'])
)
display(screening_summary)

screening_winner = screening_summary.iloc[0]
winner_name = screening_winner['experiment']
selected_rsm = float(screening_winner['rsm'])
print(f'Победитель screening: {winner_name}; selected_rsm={selected_rsm:g}')

,experiment,rsm,mean_rmsle,worst_rmsle,mean_best_iteration
3,rsm_1_reference,1.00,1.709172,1.714746,1012.0
0,rsm_0_50,0.50,1.709710,1.714427,1379.5
1,rsm_0_70,0.70,1.709976,1.714723,965.5
2,rsm_0_85,0.85,1.710161,1.714715,1195.5


Победитель screening: rsm_1_reference; selected_rsm=1


## 5. Полная четырёхфолдовая проверка

Если выигрывает новое значение, оно обучается на всех четырёх holdout с сохранением OOF-предсказаний. Если побеждает `rsm=1.0`, используются ранее рассчитанные результаты reference.

In [6]:
full_metrics_path = CATBOOST_RSM_DIR / 'winner_multifold_metrics.csv'
full_oof_path = CATBOOST_RSM_DIR / 'winner_oof.parquet'
is_reference = winner_name == 'rsm_1_reference'

if RUN_FULL_VALIDATION and not is_reference:
    full_metrics, full_oof = run_temporal_experiment(
        snapshots=snapshots,
        folds=all_folds,
        features=features,
        model_factory=partial(
            make_validation_model,
            **FIXED_PARAMS,
            rsm=selected_rsm,
        ),
        keep_oof=True,
        label=winner_name,
    )
    full_metrics['rsm'] = selected_rsm
    full_metrics.to_csv(full_metrics_path, index=False)
    full_oof.to_parquet(full_oof_path, index=False)
elif is_reference:
    full_metrics = reference_metrics.copy()
    full_oof = reference_oof.copy()
elif full_metrics_path.exists() and full_oof_path.exists():
    full_metrics = pd.read_csv(full_metrics_path)
    full_oof = pd.read_parquet(full_oof_path)
else:
    full_metrics = None
    full_oof = None
    print('Полная проверка ещё не выполнена.')

## 6. Сравнение с reference

Global OOF RMSLE является главным итогом. Per-fold таблица нужна, чтобы определить устойчивость изменения во времени.

In [7]:
if full_metrics is not None:
    fold_comparison = reference_metrics[[
        'validation_anchor', 'catboost_rmsle'
    ]].rename(columns={'catboost_rmsle': 'reference_rmsle'}).merge(
        full_metrics[['validation_anchor', 'catboost_rmsle', 'best_iteration']],
        on='validation_anchor',
        how='inner',
    ).rename(columns={'catboost_rmsle': 'candidate_rmsle'})
    fold_comparison['improvement'] = (
        fold_comparison['reference_rmsle']
        - fold_comparison['candidate_rmsle']
    )
    display(fold_comparison)

    winner_oof_rmsle = rmsle(
        full_oof['target'].to_numpy(),
        full_oof['prediction'].to_numpy(),
    )
    global_improvement = reference_oof_rmsle - winner_oof_rmsle
    print(f'Выбранный rsm:             {selected_rsm:g}')
    print(f'Reference global OOF:      {reference_oof_rmsle:.6f}')
    print(f'Candidate global OOF:      {winner_oof_rmsle:.6f}')
    print(f'Улучшение:                 {global_improvement:+.6f}')

,validation_anchor,reference_rmsle,candidate_rmsle,best_iteration,improvement
0,2025-10-22,1.714746,1.714746,1302,0.0
1,2025-11-19,1.752591,1.752591,1315,0.0
2,2025-12-17,1.752772,1.752772,1234,0.0
3,2026-01-14,1.703598,1.703598,722,0.0


Выбранный rsm:             1
Reference global OOF:      1.731068
Candidate global OOF:      1.731068
Улучшение:                 +0.000000


## 7. Итог эксперимента

| rsm | Примерно доступно признаков | RMSLE 22.10 | RMSLE 14.01 | Средний RMSLE |
|---:|---:|---:|---:|---:|
| 1.00 | 216 | 1.714746 | 1.703598 | **1.709172** |
| 0.50 | 108 | 1.714427 | 1.704992 | 1.709710 |
| 0.70 | 151 | 1.714723 | 1.705229 | 1.709976 |
| 0.85 | 184 | 1.714715 | 1.705607 | 1.710161 |

Все новые значения немного улучшили октябрьский фолд, но ухудшили январский. Лучшим новым кандидатом стал `rsm=0.50`: относительно reference он улучшил октябрь на `0.000319`, но ухудшил январь на `0.001394`, поэтому его средний RMSLE оказался хуже на `0.000538`.

При `rsm=0.50` январская модель дошла до 1636-й итерации и почти исчерпала лимит 1800 деревьев. Следовательно, её проигрыш не вызван преждевременным early stopping: ограничение доступных признаков действительно оказалось невыгодным на позднем фолде.

Reference `rsm=1.0` сохранился. Рабочая конфигурация остаётся `depth=6`, `l2_leaf_reg=10`, `random_strength=0.5`, `rsm=1.0`, а global OOF RMSLE — `1.731068`. Новая четырёхфолдовая проверка не потребовалась, поскольку победила уже полностью проверенная конфигурация.

Все флаги выше выключены. Обычный перезапуск загружает отдельные сохранённые CSV каждого кандидата без повторного обучения. Новый submission в этом ноутбуке не создаётся.